<a href="https://colab.research.google.com/github/Pranayshukla0610/ML-projects-portfolio/blob/main/HSBC_FINANCIAL_INTELLIGENCE_DASHBOARD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install bokeh yfinance pandas numpy scipy selenium pillow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 14.5 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0


In [4]:
import yfinance as yf
import pandas as pd
import numpy as np
from bokeh.io import (
    output_notebook,
    show,
    curdoc,
    output_file,
    export_png
)

from bokeh.layouts import (
    row,
    column
)
from bokeh.plotting import figure
from bokeh.models import (
    ColumnDataSource,
    HoverTool,
    Div,
    Select,
    DateRangeSlider
)
from google.colab import files

output_notebook()

In [5]:
BACKGROUND = '#0F172A'

CARD = '#1E293B'

TEXT = '#F8FAFC'

GRID = '#334155'

WIDTH = 650

HEIGHT = 400

In [6]:
companies = {
    "HSBC":"HSBC",
    "Goldman Sachs":"GS",
    "JP Morgan":"JPM",
    "Morgan Stanley":"MS",
    "Citigroup":"C"
}

In [7]:
ticker = companies['HSBC']

In [8]:
df = yf.download(
    ticker,
    period='2y',
    interval='1d'
)

/tmp/ipykernel_1530/843910617.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
[*********************100%***********************]  1 of 1 completed


In [9]:
df.head()

Price,Close,High,Low,Open,Volume
Ticker,HSBC,HSBC,HSBC,HSBC,HSBC
Date,,,,,
2024-05-16,39.809811,40.669981,39.272205,40.669981,4597700
2024-05-17,39.908375,40.015895,39.711251,39.756051,1125800
2024-05-20,39.594769,39.899413,39.585810,39.854613,977300
2024-05-21,39.944214,39.971093,39.693328,39.738128,1208400
2024-05-22,39.612690,39.962134,39.514128,39.881493,1181200


In [10]:
df.reset_index(inplace=True)

In [11]:
df.columns = df.columns.get_level_values(0)

In [12]:
df['Return'] = (
    df['Close'].pct_change()
)

In [13]:
df['MA20'] = (
    df['Close']
    .rolling(20)
    .mean()
)

df['MA50'] = (
    df['Close']
    .rolling(50)
    .mean()
)

df['MA100'] = (
    df['Close']
    .rolling(100)
    .mean()
)

In [14]:
df['Volatility'] = (
    df['Return']
    .rolling(20)
    .std()
)

In [15]:
delta = df['Close'].diff()

gain = delta.where(
    delta > 0,
    0
)

loss = -delta.where(
    delta < 0,
    0
)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df['RSI'] = (
    100 - (100 / (1 + rs))
)

In [18]:
investment = 10000

shares = (

    investment
    /
    df['Close'].iloc[0]

)

df['Portfolio'] = (

    shares
    * df['Close']

)

In [19]:
df.dropna(inplace=True)

In [20]:
source = ColumnDataSource(df)

In [21]:
latest_price = round(
    df['Close'].iloc[-1],
    2
)

daily_return = round(
    df['Return'].iloc[-1]*100,
    2
)

volatility = round(
    df['Volatility'].mean() * 100
)

portfolio_value = round(
    df['Portfolio'].iloc[-1],
    2
)

profit = round(
    portfolio_value - investment,
    2
)

In [24]:
risk_free_rate = 0.02

sharpe_ratio = round(

    (
        df['Return'].mean()*252
        - risk_free_rate
    )

    /

    (
        df['Return'].std()
        * np.sqrt(252)
    ),

    2
)

In [25]:
def create_kpi(title, value, color):

    return Div(text=f"""

    <div style="

    background:{color};

    padding:20px;

    border-radius:18px;

    text-align:center;

    width:220px;

    height:120px;

    box-shadow:0px 4px 15px rgba(0,0,0,0.5);

    ">

    <h2 style="
    color:white;
    ">

    {title}

    </h2>

    <h1 style="
    color:white;
    ">

    {value}

    </h1>

    </div>

    """)

In [26]:
kpi1 = create_kpi(
    "Live Price",
    f"${latest_price}",
    "#2563EB"
)

kpi2 = create_kpi(
    "Daily Return",
    f"{daily_return}%",
    "#059669"
)

kpi3 = create_kpi(
    "Volatility",
    f"{volatility}%",
    "#DC2626"
)

kpi4 = create_kpi(
    "Portfolio Value",
    f"${portfolio_value}",
    "#7C3AED"
)

kpi5 = create_kpi(
    "Profit/Loss",
    f"${profit}",
    "#EA580C"
)

kpi6 = create_kpi(
    "Sharpe Ratio",
    sharpe_ratio,
    "#0891B2"
)

In [27]:
company_selector = Select(

    title="Select Bank",

    value="HSBC",

    options=list(companies.keys())

)

In [29]:
date_slider = DateRangeSlider(

    title="Select Date Range",

    start=df['Date'].min(),

    end=df['Date'].max(),

    value=(
        df['Date'].min(),
        df['Date'].max()
    ),

    width=500

)

In [30]:
inc = df['Close'] > df['Open']
dec = df['Open'] > df['Close']

w = 12 * 60 * 60 * 1000

In [31]:
p1 = figure(

    x_axis_type="datetime",

    width=WIDTH,

    height=HEIGHT,

    title="HSBC Candlestick Analytics",

    background_fill_color=CARD,

    border_fill_color=BACKGROUND

)

In [32]:
p1.segment (
    df['Date'],
    df['High'],

    df['Date'],
    df['Low'],

    color = 'white'
)

GlyphRenderer(id='p1074', ...)

In [33]:
p1.vbar(

    x=df.loc[inc,'Date'],

    width=w,

    top=df.loc[inc,'Close'],

    bottom=df.loc[inc,'Open'],

    fill_color="#10B981",

    line_color="#10B981"

)

GlyphRenderer(id='p1085', ...)

In [35]:
p1.vbar(

    x=df.loc[dec,'Date'],

    width=w,

    top=df.loc[dec,'Open'],

    bottom=df.loc[dec,'Close'],

    fill_color="#EF4444",

    line_color="#EF4444"

)

GlyphRenderer(id='p1096', ...)

In [36]:
p1.vbar(

    x=df.loc[dec,'Date'],

    width=w,

    top=df.loc[dec,'Open'],

    bottom=df.loc[dec,'Close'],

    fill_color="#EF4444",

    line_color="#EF4444"

)

GlyphRenderer(id='p1107', ...)

In [37]:
p2 = figure(

    x_axis_type="datetime",

    width=WIDTH,

    height=HEIGHT,

    title="Moving Average Analytics",

    background_fill_color=CARD,

    border_fill_color=BACKGROUND

)

In [38]:
p2.line(

    df['Date'],

    df['Close'],

    line_width=2,

    color="white",

    legend_label="Close Price"

)

GlyphRenderer(id='p1167', ...)

In [39]:
p2.line(

    df['Date'],

    df['MA20'],

    line_width=3,

    color="#3B82F6",

    legend_label="MA20"

)

GlyphRenderer(id='p1180', ...)

In [40]:
p2.line(

    df['Date'],

    df['MA50'],

    line_width=3,

    color="#F59E0B",

    legend_label="MA50"

)

GlyphRenderer(id='p1192', ...)

In [41]:
p2.line(

    df['Date'],

    df['MA100'],

    line_width=3,

    color="#8B5CF6",

    legend_label="MA100"

)

GlyphRenderer(id='p1204', ...)

In [42]:
p3 = figure(

    x_axis_type="datetime",

    width=WIDTH,

    height=HEIGHT,

    title="Trading Volume Analytics",

    background_fill_color=CARD,

    border_fill_color=BACKGROUND

)

In [43]:
p3.vbar(

    x=df['Date'],

    top=df['Volume'],

    width=w,

    color="#06B6D4"

)

GlyphRenderer(id='p1265', ...)

In [44]:
p4 = figure(

    x_axis_type="datetime",

    width=WIDTH,

    height=HEIGHT,

    title="RSI Momentum Dashboard",

    background_fill_color=CARD,

    border_fill_color=BACKGROUND

)

In [45]:
p4.line(

    df['Date'],

    df['RSI'],

    line_width=3,

    color="#F97316"

)

GlyphRenderer(id='p1325', ...)

In [47]:
p4.line(

    df['Date'],

    [70]*len(df),

    color="red",

    line_dash="dashed"

)

p4.line(

    df['Date'],

    [30]*len(df),

    color="green",

    line_dash="dashed"

)

GlyphRenderer(id='p1345', ...)

In [48]:
p5 = figure(

    x_axis_type="datetime",

    width=WIDTH,

    height=HEIGHT,

    title="Portfolio Growth Dashboard",

    background_fill_color=CARD,

    border_fill_color=BACKGROUND

)

In [49]:
p5.line(

    df['Date'],

    df['Portfolio'],

    line_width=4,

    color="#22C55E"

)

GlyphRenderer(id='p1405', ...)

In [51]:
hist, edges = np.histogram(

    df['Return'].dropna(),

    bins=40

)

In [52]:
p6 = figure(

    width=WIDTH,

    height=HEIGHT,

    title="Return Distribution Analytics",

    background_fill_color=CARD,

    border_fill_color=BACKGROUND

)

In [53]:
p6.quad(

    top=hist,

    bottom=0,

    left=edges[:-1],

    right=edges[1:],

    fill_color="#8B5CF6",

    line_color="white"

)

GlyphRenderer(id='p1451', ...)

In [54]:
p7 = figure(

    x_axis_type="datetime",

    width=WIDTH,

    height=HEIGHT,

    title="Volatility Risk Analytics",

    background_fill_color=CARD,

    border_fill_color=BACKGROUND

)

In [55]:
p7.line(

    df['Date'],

    df['Volatility'],

    line_width=3,

    color="#EF4444"

)

GlyphRenderer(id='p1511', ...)

In [56]:
hover = HoverTool(

    tooltips=[

        ("Date", "@Date{%F}"),
        ("Open", "@Open"),
        ("Close", "@Close"),
        ("High", "@High"),
        ("Low", "@Low"),
        ("Volume", "@Volume")

    ],

    formatters={

        '@Date':'datetime'

    }

)

In [57]:
p1.add_tools(hover)

In [58]:
plots = [

    p1,
    p2,
    p3,
    p4,
    p5,
    p6,
    p7

]

In [59]:
for p in plots:

    p.title.text_color = TEXT

    p.xaxis.major_label_text_color = TEXT

    p.yaxis.major_label_text_color = TEXT

    p.xaxis.axis_label_text_color = TEXT

    p.yaxis.axis_label_text_color = TEXT

    p.grid.grid_line_color = GRID

In [60]:
title = Div(text="""

<div style="

background:linear-gradient(
90deg,
#111827,
#2563EB
);

padding:25px;

border-radius:18px;

box-shadow:0px 4px 15px rgba(0,0,0,0.5);

">

<h1 style="

color:white;

text-align:center;

font-size:40px;

">

HSBC FINANCIAL INTELLIGENCE DASHBOARD

</h1>

<p style="

color:white;

text-align:center;

font-size:18px;

">

Global Banking Analytics |
Risk Intelligence |
Portfolio Monitoring

</p>

</div>

""")

In [61]:
subtitle = Div(text="""

<h2 style="
color:white;
padding:10px;
">

Interactive Global Banking Analytics Platform

</h2>

""")

In [62]:
divider = Div(text="""

<hr style="
border:2px solid #334155;
">

""")

In [63]:
market_trend = "Bullish"

if daily_return < 0:

    market_trend = "Bearish"

elif daily_return == 0:

    market_trend = "Neutral"

In [64]:
market_status = Div(text=f"""

<div style="

background:#0EA5E9;

padding:20px;

border-radius:18px;

width:220px;

height:120px;

text-align:center;

box-shadow:0px 4px 15px rgba(0,0,0,0.5);

">

<h2 style="color:white;">

Market Trend

</h2>

<h1 style="color:white;">

{market_trend}

</h1>

</div>

""")

In [71]:
risk_level = "Low"

if volatility > 3:

    risk_level = "High"

elif volatility > 1:

    risk_level = "Moderate"

In [72]:
risk_card = Div(text=f"""

<div style="

background:#DC2626;

padding:20px;

border-radius:18px;

width:220px;

height:120px;

text-align:center;

box-shadow:0px 4px 15px rgba(0,0,0,0.5);

">

<h2 style="color:white;">

Risk Level

</h2>

<h1 style="color:white;">

{risk_level}

</h1>

</div>

""")

In [67]:
annual_return = round(

    df['Return'].mean()
    * 252
    * 100,

    2

)

In [68]:
return_card = Div(text=f"""

<div style="

background:#16A34A;

padding:20px;

border-radius:18px;

width:220px;

height:120px;

text-align:center;

box-shadow:0px 4px 15px rgba(0,0,0,0.5);

">

<h2 style="color:white;">

Annual Return

</h2>

<h1 style="color:white;">

{annual_return}%

</h1>

</div>

""")

In [73]:
dashboard = column(

    title,

    subtitle,

    divider,

    row(
        kpi1,
        kpi2,
        kpi3
    ),

    row(
        kpi4,
        kpi5,
        kpi6
    ),

    row(
        market_status,
        risk_card,
        return_card
    ),

    divider,

    row(
        company_selector,
        date_slider
    ),

    row(
        p1,
        p2
    ),

    row(
        p3,
        p4
    ),

    row(
        p5,
        p6
    ),

    row(
        p7
    )

)

In [74]:
show(dashboard)

ERROR:bokeh.core.validation.check:E-1027 (REPEATED_LAYOUT_CHILD): The same model can't be used multiple times in a layout: Column(id='p1533', ...)
